# Step 9 — Build daily FinBERT sentiment features

We score the strictly filtered 2023–2024 Apple headlines, assign each headline to the trading day when it was available by market close, and calculate five-trading-day sentiment features.

In [1]:
from datetime import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "ProsusAI/finbert"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "apple_news_data.csv"
PRICE_PATH = PROJECT_ROOT / "data" / "raw" / "aapl_spy_2023-01-01_to_2025-01-01.csv"
MOMENTUM_PATH = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_target.csv"
ARTICLE_OUTPUT = PROJECT_ROOT / "data" / "processed" / "apple_headlines_finbert.csv"
DAILY_OUTPUT = PROJECT_ROOT / "data" / "processed" / "apple_daily_sentiment.csv"
MERGED_OUTPUT = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_sentiment.csv"

/Users/keishakalra/Desktop/Financial_App/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Apply the validated relevance filter

We require the 2023–2024 period, exact `AAPL.US` symbol membership, and an explicit Apple or major-product reference in the title. Normalized duplicate titles are removed.

In [2]:
news = pd.read_csv(NEWS_PATH)
news["published_at"] = pd.to_datetime(news["date"], utc=True, errors="coerce")
in_period = news["published_at"].between("2023-01-01", "2025-01-01", inclusive="left")
has_aapl_symbol = news["symbols"].fillna("").str.split(",").apply(
    lambda symbols: any(symbol.strip() == "AAPL.US" for symbol in symbols)
)
apple_title_pattern = r"\b(?:Apple|AAPL|iPhone|iPad|Mac|iOS|Vision Pro)\b"
explicit_apple_title = news["title"].str.contains(
    apple_title_pattern, case=False, regex=True, na=False
)
headlines = news.loc[in_period & has_aapl_symbol & explicit_apple_title].copy()
headlines["normalized_title"] = (
    headlines["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)
headlines = (
    headlines.sort_values("published_at")
    .drop_duplicates("normalized_title", keep="first")
    .reset_index(drop=True)
)
print(f"Unique eligible headlines: {len(headlines):,}")
headlines["published_at"].dt.year.value_counts().sort_index()

Unique eligible headlines: 5,677


published_at
2023    3502
2024    2175
Name: count, dtype: int64

## Assign headlines without looking ahead

Timestamps are converted from UTC to New York time. A headline published by 4:00 p.m. Eastern on a trading day is assigned to that date. After-close, weekend, and holiday headlines move to the next trading day.

In [3]:
prices = pd.read_csv(PRICE_PATH, header=[0, 1], index_col=0, parse_dates=True)
trading_dates = pd.DatetimeIndex(prices.index).normalize().sort_values()
trading_date_set = set(trading_dates)

def assign_trading_date(timestamp):
    local_timestamp = timestamp.tz_convert("America/New_York")
    local_date = pd.Timestamp(local_timestamp.date())
    if local_date in trading_date_set and local_timestamp.time() <= time(16, 0):
        return local_date
    side = "right" if local_date in trading_date_set else "left"
    position = trading_dates.searchsorted(local_date, side=side)
    return trading_dates[position] if position < len(trading_dates) else pd.NaT

headlines["trading_date"] = headlines["published_at"].apply(assign_trading_date)
assert headlines["trading_date"].notna().all()
headlines[["published_at", "trading_date", "title"]].head()

,published_at,trading_date,title
0,2023-01-01 11:00:00+00:00,2023-01-03,"Down 28% in 2022, Is Apple Stock a Buy for 2023?"
1,2023-01-02 11:00:15+00:00,2023-01-03,39%of this Apple Inc. (NASDAQ:AAPL) insider's ...
2,2023-01-02 12:00:25+00:00,2023-01-03,Shopify in advertising push to fill void left ...
3,2023-01-02 16:20:00+00:00,2023-01-03,3 Reasons Apple Stock Keeps Falling
4,2023-01-03 03:19:45+00:00,2023-01-03,iPhone City Is Back at 90% Capacity After Covi...


## Score headlines with FinBERT

The continuous score is positive probability minus negative probability. It ranges approximately from −1 to +1.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
id_to_label = {int(key): value.lower() for key, value in model.config.id2label.items()}

all_probabilities = []
batch_size = 64
for start in range(0, len(headlines), batch_size):
    batch_titles = headlines["title"].iloc[start : start + batch_size].tolist()
    encoded = tokenizer(
        batch_titles, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    with torch.no_grad():
        batch_probabilities = torch.softmax(model(**encoded).logits, dim=1).cpu()
    all_probabilities.append(batch_probabilities)

probabilities = torch.cat(all_probabilities).numpy()
for class_id, label in id_to_label.items():
    headlines[f"finbert_{label}_probability"] = probabilities[:, class_id]
headlines["finbert_label"] = [
    id_to_label[int(class_id)].title() for class_id in probabilities.argmax(axis=1)
]
headlines["sentiment_score"] = (
    headlines["finbert_positive_probability"]
    - headlines["finbert_negative_probability"]
)
headlines["finbert_model"] = MODEL_NAME
headlines.to_csv(ARTICLE_OUTPUT, index=False)
headlines["finbert_label"].value_counts()

finbert_label
Neutral     3169
Negative    1456
Positive    1052
Name: count, dtype: int64

## Aggregate into five-trading-day features

We sum article scores and counts by assigned trading date, fill missing daily counts with zero, then roll across the current and previous four trading days. The average weights every headline equally. A window with no headlines retains a missing average rather than pretending the news was neutral.

In [5]:
article_daily = headlines.groupby("trading_date").agg(
    daily_sentiment_sum=("sentiment_score", "sum"),
    daily_headline_count=("title", "size"),
)
daily = article_daily.reindex(trading_dates, fill_value=0)
daily.index.name = "date"
daily["sentiment_sum_5d"] = daily["daily_sentiment_sum"].rolling(5, min_periods=1).sum()
daily["headline_count_5d"] = daily["daily_headline_count"].rolling(5, min_periods=1).sum().astype(int)
daily["average_sentiment_5d"] = (
    daily["sentiment_sum_5d"] / daily["headline_count_5d"].replace(0, pd.NA)
)
daily.to_csv(DAILY_OUTPUT)
daily.head(10)

,daily_sentiment_sum,daily_headline_count,sentiment_sum_5d,headline_count_5d,average_sentiment_5d
date,,,,,
2023-01-03,-14.236697,27,-14.236697,27,-0.527285
2023-01-04,-6.671336,19,-20.908033,46,-0.454522
2023-01-05,-1.251860,16,-22.159893,62,-0.357418
2023-01-06,-0.763297,6,-22.923190,68,-0.337106
2023-01-09,2.839273,20,-20.083916,88,-0.228226
2023-01-10,-2.774816,22,-8.622035,83,-0.10388
2023-01-11,-2.457413,22,-4.408112,86,-0.051257
2023-01-12,-1.274364,13,-4.430616,83,-0.053381
2023-01-13,-7.309470,16,-10.976789,93,-0.11803


In [6]:
momentum = pd.read_csv(MOMENTUM_PATH, parse_dates=["date"], index_col="date")
combined = momentum.join(daily[["average_sentiment_5d", "headline_count_5d"]], how="left")
combined.to_csv(MERGED_OUTPUT)

print(f"Saved {len(headlines):,} article scores to {ARTICLE_OUTPUT}")
print(f"Saved {len(daily):,} daily rows to {DAILY_OUTPUT}")
print(f"Saved {len(combined):,} combined rows to {MERGED_OUTPUT}")
pd.Series({
    "Combined rows": len(combined),
    "Rows with recent headlines": combined["headline_count_5d"].gt(0).sum(),
    "Rows without recent headlines": combined["headline_count_5d"].eq(0).sum(),
    "Average five-day headline count": combined["headline_count_5d"].mean(),
})

Saved 5,677 article scores to /Users/keishakalra/Desktop/Financial_App/data/processed/apple_headlines_finbert.csv
Saved 502 daily rows to /Users/keishakalra/Desktop/Financial_App/data/processed/apple_daily_sentiment.csv
Saved 492 combined rows to /Users/keishakalra/Desktop/Financial_App/data/processed/aapl_momentum_sentiment.csv


Combined rows                      492.000000
Rows with recent headlines         477.000000
Rows without recent headlines       15.000000
Average five-day headline count     57.101626
dtype: float64